# Notebook 3A — Advanced NLP Pipeline
### Fake Internship & Job Scam Detection System

**Purpose:**  
This notebook constructs a full traditional NLP pipeline on raw job listing text columns.  
It operates independently of Notebook 3 (Feature Engineering) and produces two artefacts:  
- `../data/nlp_feature_dataset.csv` — per-listing NLP features  
- `../data/tfidf_features.csv` — TF-IDF vectorized description matrix  

**Pipeline Overview:**

| Step | Operation | Output |
|------|-----------|--------|
| 1 | Import Libraries | — |
| 2 | Load Dataset | `df` |
| 3 | Text Cleaning | `clean_description` |
| 4 | Tokenization | `tokenized_description` |
| 5 | Stopword Removal | `filtered_tokens` |
| 6 | Lemmatization | `lemmatized_text` |
| 7 | N-Gram Generation | `top_bigrams`, `top_trigrams` |
| 8 | TF-IDF Extraction | `tfidf_matrix`, `tfidf_features.csv` |
| 9 | Scam Phrase Analysis | `scam_phrase_frequency_table` |
| 10 | Readability Analysis | `readability_score` |
| 11 | Text Length Analytics | `word_count`, `sentence_count`, `average_word_length` |
| 12 | Vocabulary Analysis | `unique_word_count`, `lexical_diversity` |
| 13 | NLP Feature Dataset | `nlp_feature_dataset.csv` |
| 14 | Export Results | Both CSVs confirmed |

> **Constraint:** No deep learning models (BERT, Word2Vec, etc.) are used.  
> This is a production-grade **traditional NLP** pipeline.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# --- NLTK ---
import nltk
nltk.download('stopwords',   quiet=True)
nltk.download('punkt',       quiet=True)
nltk.download('punkt_tab',   quiet=True)
from nltk.corpus import stopwords
from nltk.util  import ngrams
from collections import Counter

# --- spaCy ---
import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

# --- scikit-learn ---
from sklearn.feature_extraction.text import TfidfVectorizer

# --- textstat ---
try:
    from textstat import flesch_reading_ease, sentence_count as ts_sentence_count
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'textstat', '-q'])
    from textstat import flesch_reading_ease, sentence_count as ts_sentence_count

# --- visualisation ---
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for all environments
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ENGLISH_STOPWORDS = set(stopwords.words('english'))

print("All libraries loaded successfully.")
print(f"  spaCy model : en_core_web_sm  v{nlp.meta['version']}")
print(f"  Stopwords   : {len(ENGLISH_STOPWORDS)} words loaded from NLTK")

All libraries loaded successfully.
  spaCy model : en_core_web_sm  v3.8.0
  Stopwords   : 198 words loaded from NLTK


In [2]:
# STEP 2: LOAD DATASET

df = pd.read_csv('../data/processed_cleaned_data.csv')

# --- Safe initialisation of all text columns ---
TEXT_COLUMNS = ['title', 'description', 'skills', 'company_about']
for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna('Missing').astype(str)
    else:
        df[col] = 'Missing'   # Create column if absent

# --- Preserve is_scam label for reference (not used in NLP transforms) ---
if 'is_scam' not in df.columns:
    df['is_scam'] = np.nan

# --- Basic verification ---
print("Dataset loaded successfully.")
print(f"  Shape            : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Scam label dist  : {df['is_scam'].value_counts().to_dict()}")
print("\nText column null counts (post-fill):")
print(df[TEXT_COLUMNS].isnull().sum())
print("\nSample description (first record):")
print(df['description'].iloc[0][:300], '...')

Dataset loaded successfully.
  Shape            : 13,616 rows × 25 columns
  Scam label dist  : {0: 10607, 1: 3009}

Text column null counts (post-fill):
title            0
description      0
skills           0
company_about    0
dtype: int64

Sample description (first record):
Selected intern's day-to-day responsibilities include: 1. Assist in creating detailed reports for buy/hold/sell recommendations. 2. Help in tracking portfolio performance. 3. Collect, analyze, and interpret financial data from companies, sectors, and industries. 4. Track stock markets, economic indi ...


In [3]:
# STEP 3: TEXT CLEANING

def clean_text(text: str) -> str:
    """Apply a deterministic 6-stage cleaning pipeline to raw text."""
    # Stage 1: Lowercase
    text = text.lower()
    # Stage 2: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Stage 3: Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)
    # Stage 4: Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Stage 5: Collapse multiple whitespace
    text = re.sub(r'\s+', ' ', text)
    # Stage 6: Strip boundaries
    return text.strip()

df['clean_description'] = df['description'].apply(clean_text)

# --- Sanity check ---
print("STEP 3 complete — clean_description generated.")
print("\nBefore cleaning:")
print(df['description'].iloc[0][:250])
print("\nAfter cleaning:")
print(df['clean_description'].iloc[0][:250])

STEP 3 complete — clean_description generated.

Before cleaning:
Selected intern's day-to-day responsibilities include: 1. Assist in creating detailed reports for buy/hold/sell recommendations. 2. Help in tracking portfolio performance. 3. Collect, analyze, and interpret financial data from companies, sectors, and

After cleaning:
selected intern s day to day responsibilities include assist in creating detailed reports for buy hold sell recommendations help in tracking portfolio performance collect analyze and interpret financial data from companies sectors and industries trac


In [4]:
# STEP 4: TOKENIZATION

print("Running spaCy tokenization (batch mode)...")

# Disable heavy pipeline components not needed for tokenization
BATCH_SIZE = 500

tokenized_all = []
for doc in nlp.pipe(df['clean_description'].tolist(),
                    batch_size=BATCH_SIZE,
                    disable=['parser', 'ner']):
    tokens = [token.text for token in doc
              if token.is_alpha and len(token.text) >= 2]
    tokenized_all.append(tokens)

df['tokenized_description'] = tokenized_all

print("STEP 4 complete — tokenized_description generated.")
print(f"  Avg tokens per description : {df['tokenized_description'].apply(len).mean():.1f}")
print("\nExample:")
print(f"  Input : {df['clean_description'].iloc[0][:80]}...")
print(f"  Output: {df['tokenized_description'].iloc[0][:12]}")

Running spaCy tokenization (batch mode)...
STEP 4 complete — tokenized_description generated.
  Avg tokens per description : 230.7

Example:
  Input : selected intern s day to day responsibilities include assist in creating detaile...
  Output: ['selected', 'intern', 'day', 'to', 'day', 'responsibilities', 'include', 'assist', 'in', 'creating', 'detailed', 'reports']


In [5]:
# STEP 5: STOPWORD REMOVAL

def remove_stopwords(tokens: list) -> list:
    """Filter NLTK English stopwords from a token list."""
    return [t for t in tokens if t not in ENGLISH_STOPWORDS]

df['filtered_tokens'] = df['tokenized_description'].apply(remove_stopwords)

# --- Compression ratio: how many tokens removed on average ---
avg_before = df['tokenized_description'].apply(len).mean()
avg_after  = df['filtered_tokens'].apply(len).mean()
reduction  = (1 - avg_after / avg_before) * 100 if avg_before > 0 else 0

print("STEP 5 complete — filtered_tokens generated.")
print(f"  Avg tokens before removal : {avg_before:.1f}")
print(f"  Avg tokens after removal  : {avg_after:.1f}")
print(f"  Vocabulary reduction      : {reduction:.1f}%")
print("\nExample:")
print(f"  Before: {df['tokenized_description'].iloc[0][:10]}")
print(f"  After : {df['filtered_tokens'].iloc[0][:10]}")

STEP 5 complete — filtered_tokens generated.
  Avg tokens before removal : 230.7
  Avg tokens after removal  : 163.5
  Vocabulary reduction      : 29.1%

Example:
  Before: ['selected', 'intern', 'day', 'to', 'day', 'responsibilities', 'include', 'assist', 'in', 'creating']
  After : ['selected', 'intern', 'day', 'day', 'responsibilities', 'include', 'assist', 'creating', 'detailed', 'reports']


In [6]:
# STEP 6: LEMMATIZATION

print("Running spaCy lemmatization (batch mode)...")

# Re-join filtered tokens and pass through spaCy for lemmatization
filtered_texts = df['filtered_tokens'].apply(lambda tokens: ' '.join(tokens)).tolist()

lemmatized_all = []
for doc in nlp.pipe(filtered_texts,
                    batch_size=BATCH_SIZE,
                    disable=['parser', 'ner']):
    lemmas = [token.lemma_ for token in doc
              if token.is_alpha
              and len(token.lemma_) >= 2
              and token.lemma_ not in ENGLISH_STOPWORDS]
    lemmatized_all.append(' '.join(lemmas))

df['lemmatized_text'] = lemmatized_all

print("STEP 6 complete — lemmatized_text generated.")
print("\nExample:")
print(f"  Filtered tokens : {df['filtered_tokens'].iloc[0][:8]}")
print(f"  Lemmatized text : {df['lemmatized_text'].iloc[0][:120]}")

Running spaCy lemmatization (batch mode)...
STEP 6 complete — lemmatized_text generated.

Example:
  Filtered tokens : ['selected', 'intern', 'day', 'day', 'responsibilities', 'include', 'assist', 'creating']
  Lemmatized text : select intern day day responsibility include assist create detailed report buy hold sell recommendation help track portf


In [7]:
# STEP 7: N-GRAM GENERATION

# --- Collect all bi-grams across corpus ---
all_bigrams  = []
all_trigrams = []

for tokens in df['filtered_tokens']:
    all_bigrams.extend(ngrams(tokens, 2))
    all_trigrams.extend(ngrams(tokens, 3))

bigram_freq  = Counter(all_bigrams)
trigram_freq = Counter(all_trigrams)

# --- Top 20 results ---
top_bigrams  = bigram_freq.most_common(20)
top_trigrams = trigram_freq.most_common(20)

print("STEP 7 complete — N-gram analysis done.")
print(f"  Total unique bi-grams  : {len(bigram_freq):,}")
print(f"  Total unique tri-grams : {len(trigram_freq):,}")

print("\nTop 20 Bi-grams:")
for gram, freq in top_bigrams:
    print(f"  {' '.join(gram):<35} {freq:>6}")

print("\nTop 20 Tri-grams:")
for gram, freq in top_trigrams:
    print(f"  {' '.join(gram):<45} {freq:>6}")

STEP 7 complete — N-gram analysis done.
  Total unique bi-grams  : 389,449
  Total unique tri-grams : 705,821

Top 20 Bi-grams:
  social media                          3917
  nbsp nbsp                             3849
  day day                               3601
  key responsibilities                  3455
  responsibilities include              3296
  work home                             3237
  communication skills                  2689
  years experience                      2542
  day responsibilities                  2464
  selected intern                       2460
  intern day                            2456
  data entry                            2316
  digital marketing                     1980
  full time                             1892
  job description                       1841
  hands experience                      1695
  attention detail                      1585
  fast paced                            1577
  bachelor degree                       1545
  home opportunit

In [8]:
# --- Visualise top bi-grams and tri-grams ---

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Top 20 N-Grams — Job Description Corpus', fontsize=15, fontweight='bold')

# Bi-gram chart
bg_labels = [' '.join(g) for g, _ in top_bigrams]
bg_counts = [c for _, c in top_bigrams]
axes[0].barh(bg_labels[::-1], bg_counts[::-1], color='steelblue')
axes[0].set_title('Top 20 Bi-grams', fontsize=13)
axes[0].set_xlabel('Frequency')
axes[0].tick_params(axis='y', labelsize=9)

# Tri-gram chart
tg_labels = [' '.join(g) for g, _ in top_trigrams]
tg_counts = [c for _, c in top_trigrams]
axes[1].barh(tg_labels[::-1], tg_counts[::-1], color='coral')
axes[1].set_title('Top 20 Tri-grams', fontsize=13)
axes[1].set_xlabel('Frequency')
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../data/ngram_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("N-gram chart saved to ../data/ngram_analysis.png")

N-gram chart saved to ../data/ngram_analysis.png


In [9]:
# STEP 8: TF-IDF FEATURE EXTRACTION

tfidf_vectorizer = TfidfVectorizer(
    max_features = 200,
    ngram_range  = (1, 2),
    min_df       = 5,
    sublinear_tf = True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['lemmatized_text'])

feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df      = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# --- Top terms by mean TF-IDF score across corpus ---
top_terms = pd.Series(
    tfidf_matrix.toarray().mean(axis=0),
    index=feature_names
).sort_values(ascending=False).head(25)

print("STEP 8 complete — TF-IDF matrix generated.")
print(f"  Matrix shape   : {tfidf_matrix.shape}  (docs × features)")
print(f"  Sparsity       : {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.1f}%")
print("\nTop 25 terms by mean corpus TF-IDF score:")
print(top_terms.to_string())

STEP 8 complete — TF-IDF matrix generated.
  Matrix shape   : (13616, 200)  (docs × features)
  Sparsity       : 81.3%

Top 25 terms by mean corpus TF-IDF score:
work              0.076427
day               0.058524
home              0.055910
team              0.055461
experience        0.055009
work home         0.054739
opportunity       0.053714
intern            0.053348
apply             0.047901
include           0.045673
skill             0.045526
support           0.044824
internship        0.044490
responsibility    0.042585
job               0.040236
customer          0.040032
ensure            0.039827
maintain          0.038659
datum             0.038540
role              0.038117
client            0.037210
sale              0.036530
business          0.036011
management        0.035892
hr                0.035509


In [10]:
# --- Visualise top TF-IDF terms ---

fig, ax = plt.subplots(figsize=(12, 7))
top_terms.sort_values().plot(kind='barh', ax=ax, color='mediumseagreen')
ax.set_title('Top 25 Terms — Mean TF-IDF Score (Corpus)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean TF-IDF Score')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('../data/tfidf_top_terms.png', dpi=120, bbox_inches='tight')
plt.show()
print("TF-IDF chart saved to ../data/tfidf_top_terms.png")

TF-IDF chart saved to ../data/tfidf_top_terms.png


In [11]:
# STEP 9: SCAM PHRASE ANALYSIS

SCAM_PHRASES = [
    'registration fee',  'security deposit',  'processing fee',
    'training fee',      'training charge',   'refundable deposit',
    'telegram',          'whatsapp',           'earn online',
    'investment',        'pay to work',        'income source',
    'typing job',        'form filling',       'work from home job',
    'part time job',     'daily payment',      'earn money fast',
    'no experience',     'immediate joining',  'limited seats',
    'call now',          'apply immediately',  'data entry',
    'ad posting',        'copy paste job',     'captcha job',
    'investment required','refer and earn',    'multi level',
]

desc_lower = df['description'].str.lower()
has_label = bool(df['is_scam'].notna().any()) and int(df['is_scam'].nunique()) > 1

rows = []
for phrase in SCAM_PHRASES:
    mask         = desc_lower.str.contains(re.escape(phrase), regex=True)
    total_count  = mask.sum()
    prevalence   = (total_count / len(df)) * 100

    scam_count = legit_count = np.nan
    if has_label:
        scam_count  = mask[df['is_scam'] == 1].sum()
        legit_count = mask[df['is_scam'] == 0].sum()

    rows.append({
        'phrase'          : phrase,
        'total_listings'  : total_count,
        'prevalence_pct'  : round(prevalence, 3),
        'scam_listings'   : scam_count,
        'legit_listings'  : legit_count
    })

scam_phrase_frequency_table = (
    pd.DataFrame(rows)
    .sort_values('total_listings', ascending=False)
    .reset_index(drop=True)
)

print("STEP 9 complete — scam_phrase_frequency_table generated.")
print(scam_phrase_frequency_table.to_string(index=False))

STEP 9 complete — scam_phrase_frequency_table generated.
             phrase  total_listings  prevalence_pct  scam_listings  legit_listings
  apply immediately            1522          11.178            889             633
         data entry            1521          11.171            582             939
           whatsapp            1477          10.848            525             952
   registration fee             890           6.536            890               0
      limited seats             740           5.435            387             353
         investment             441           3.239            200             241
 work from home job             175           1.285              9             166
      part time job             158           1.160              0             158
       form filling             154           1.131             81              73
  immediate joining              36           0.264              6              30
         typing job           

In [12]:
# --- Visualise scam phrase prevalence ---

plot_data = scam_phrase_frequency_table[scam_phrase_frequency_table['total_listings'] > 0]

fig, ax = plt.subplots(figsize=(13, 8))
plot_data.sort_values('total_listings').plot(
    x='phrase', y='total_listings', kind='barh',
    ax=ax, color='tomato', legend=False
)
ax.set_title('Scam Phrase Frequency in Job Descriptions', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Listings Containing Phrase')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('../data/scam_phrase_frequency.png', dpi=120, bbox_inches='tight')
plt.show()
print("Chart saved to ../data/scam_phrase_frequency.png")

Chart saved to ../data/scam_phrase_frequency.png


In [13]:
# STEP 10: READABILITY ANALYSIS

df['readability_score'] = df['description'].apply(flesch_reading_ease)

stats = df['readability_score'].describe()
print("STEP 10 complete — readability_score generated.")
print(f"  Mean   : {stats['mean']:.2f}")
print(f"  Median : {df['readability_score'].median():.2f}")
print(f"  Std    : {stats['std']:.2f}")
print(f"  Min    : {stats['min']:.2f}")
print(f"  Max    : {stats['max']:.2f}")

# --- Distribution plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['readability_score'].clip(-50, 120), bins=60, color='slateblue', edgecolor='white', linewidth=0.4)
ax.axvline(df['readability_score'].mean(), color='red', linestyle='--', label=f"Mean = {df['readability_score'].mean():.1f}")
ax.axvline(df['readability_score'].median(), color='orange', linestyle='--', label=f"Median = {df['readability_score'].median():.1f}")
ax.set_title('Flesch Reading Ease Score Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Readability Score (0 = Hard, 100 = Easy)')
ax.set_ylabel('Number of Listings')
ax.legend()
plt.tight_layout()
plt.savefig('../data/readability_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Chart saved to ../data/readability_distribution.png")

STEP 10 complete — readability_score generated.
  Mean   : 16.49
  Median : 21.68
  Std    : 27.68
  Min    : -433.97
  Max    : 100.24
Chart saved to ../data/readability_distribution.png


In [14]:
# STEP 11: TEXT LENGTH ANALYTICS

def average_word_length(text: str) -> float:
    """Compute mean character length of whitespace-split tokens."""
    words = text.split()
    return np.mean([len(w) for w in words]) if words else 0.0

df['word_count']          = df['clean_description'].apply(lambda x: len(x.split()))
df['sentence_count']      = df['description'].apply(ts_sentence_count)
df['average_word_length'] = df['clean_description'].apply(average_word_length)

print("STEP 11 complete — text length analytics generated.")
print(df[['word_count', 'sentence_count', 'average_word_length']].describe().round(2).to_string())

STEP 11 complete — text length analytics generated.
       word_count  sentence_count  average_word_length
count    13616.00        13616.00             13616.00
mean       236.34           12.73                 6.09
std        266.59           13.66                 0.60
min          1.00            1.00                 3.23
25%         29.00            4.00                 5.69
50%        140.00            8.00                 6.00
75%        389.00           18.00                 6.46
max       2252.00          147.00                12.00


In [15]:
# STEP 12: VOCABULARY ANALYSIS

def unique_word_count(text: str) -> int:
    return len(set(text.split()))

def lexical_diversity(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)

df['unique_word_count'] = df['clean_description'].apply(unique_word_count)
df['lexical_diversity'] = df['clean_description'].apply(lexical_diversity)

print("STEP 12 complete — vocabulary features generated.")
print(df[['unique_word_count', 'lexical_diversity']].describe().round(4).to_string())

# --- Lexical diversity distribution plot ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['lexical_diversity'], bins=50, color='darkcyan', edgecolor='white', linewidth=0.4)
ax.set_title('Lexical Diversity Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Lexical Diversity (unique / total words)')
ax.set_ylabel('Number of Listings')
plt.tight_layout()
plt.savefig('../data/lexical_diversity_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

STEP 12 complete — vocabulary features generated.
       unique_word_count  lexical_diversity
count         13616.0000         13616.0000
mean            124.3138             0.7116
std             123.9455             0.2098
min               1.0000             0.1324
25%              26.0000             0.5660
50%              93.0000             0.6964
75%             180.0000             0.8966
max             696.0000             1.0000


In [16]:

# STEP 13: NLP FEATURE DATASET

NLP_FEATURE_COLUMNS = [
    'is_scam',
    'clean_description',
    'lemmatized_text',
    'readability_score',
    'word_count',
    'sentence_count',
    'average_word_length',
    'unique_word_count',
    'lexical_diversity',
]

nlp_feature_dataset = df[NLP_FEATURE_COLUMNS].copy()
nlp_feature_dataset = nlp_feature_dataset.fillna(0)

# --- Summary ---
print("--- NLP FEATURE DATASET VERIFICATION ---")
print(f"Shape          : {nlp_feature_dataset.shape[0]:,} rows × {nlp_feature_dataset.shape[1]} columns")
print(f"Remaining NaNs : {nlp_feature_dataset.isna().sum().sum()}")
print("\nNumeric feature summary:")
print(nlp_feature_dataset.select_dtypes(include=np.number).describe().round(3).to_string())

--- NLP FEATURE DATASET VERIFICATION ---
Shape          : 13,616 rows × 9 columns
Remaining NaNs : 0

Numeric feature summary:
         is_scam  readability_score  word_count  sentence_count  average_word_length  unique_word_count  lexical_diversity
count  13616.000          13616.000   13616.000       13616.000            13616.000          13616.000          13616.000
mean       0.221             16.486     236.337          12.727                6.085            124.314              0.712
std        0.415             27.680     266.595          13.658                0.598            123.945              0.210
min        0.000           -433.970       1.000           1.000                3.231              1.000              0.132
25%        0.000              8.046      29.000           4.000                5.692             26.000              0.566
50%        0.000             21.682     140.000           8.000                6.000             93.000              0.696
75%        0

In [17]:

# STEP 14: EXPORT RESULTS

# --- Export 1: NLP feature dataset ---
nlp_feature_dataset.to_csv('../data/nlp_feature_dataset.csv', index=False)
print("nlp_feature_dataset.csv saved.")
print(f"  Path  : ../data/nlp_feature_dataset.csv")
print(f"  Shape : {nlp_feature_dataset.shape}")

# --- Export 2: TF-IDF features ---
tfidf_df.to_csv('../data/tfidf_features.csv', index=False)
print("\ntfidf_features.csv saved.")
print(f"  Path  : ../data/tfidf_features.csv")
print(f"  Shape : {tfidf_df.shape}")

print("\n============================================================")
print(" Notebook 3A — Advanced NLP Pipeline: COMPLETE")
print("============================================================")
print(f"  Listings processed   : {len(df):,}")
print(f"  NLP features created : {len(NLP_FEATURE_COLUMNS) - 1}  (excl. label)")
print(f"  TF-IDF features      : {tfidf_df.shape[1]}")
print("  Output files         : nlp_feature_dataset.csv, tfidf_features.csv")
print("============================================================")

nlp_feature_dataset.csv saved.
  Path  : ../data/nlp_feature_dataset.csv
  Shape : (13616, 9)

tfidf_features.csv saved.
  Path  : ../data/tfidf_features.csv
  Shape : (13616, 200)

 Notebook 3A — Advanced NLP Pipeline: COMPLETE
  Listings processed   : 13,616
  NLP features created : 8  (excl. label)
  TF-IDF features      : 200
  Output files         : nlp_feature_dataset.csv, tfidf_features.csv


In [18]:
nlp_feature_dataset.columns.tolist()

['is_scam',
 'clean_description',
 'lemmatized_text',
 'readability_score',
 'word_count',
 'sentence_count',
 'average_word_length',
 'unique_word_count',
 'lexical_diversity']

In [19]:
# ============================================================
# LOAD FEATURE ENGINEERING DATASET
# ============================================================

feature_df = pd.read_csv(
    "../data/model_ready_dataset.csv"
)

print(feature_df.shape)

(13616, 24)


In [20]:
# ============================================================
# LOAD NLP DATASET
# ============================================================

nlp_df = pd.read_csv(
    "../data/nlp_feature_dataset.csv"
)

print(nlp_df.shape)

(13616, 9)


In [21]:
# ============================================================
# NLP FEATURES TO MERGE
# ============================================================

nlp_features = [

    "readability_score",

    "word_count",

    "sentence_count",

    "average_word_length",

    "unique_word_count",

    "lexical_diversity"

]

nlp_subset = nlp_df[nlp_features]

nlp_subset.head()

,readability_score,word_count,sentence_count,average_word_length,unique_word_count,lexical_diversity
0,10.980429,43,5,6.255814,38,0.883721
1,-11.262500,26,4,6.730769,23,0.884615
2,-41.511464,493,8,6.456389,236,0.478702
3,10.181974,72,8,6.930556,58,0.805556
4,50.106306,181,17,5.254144,129,0.712707


In [22]:
# ============================================================
# MERGE NLP + FEATURE ENGINEERING
# ============================================================

final_dataset = pd.concat(

    [

        feature_df.reset_index(drop=True),

        nlp_subset.reset_index(drop=True)

    ],

    axis=1

)

print(final_dataset.shape)

(13616, 30)


In [23]:
final_dataset.columns.tolist()

['is_scam',
 'cleaned_stipend_monthly',
 'has_website',
 'has_email',
 'skills_count',
 'title_word_count',
 'desc_char_length',
 'numeric_openings',
 'is_bulk_hiring',
 'is_wfh_short_duration',
 'company_name_word_count',
 'link_count_desc',
 'contact_count_desc',
 'all_caps_ratio_desc',
 'keyword_count',
 'stipend_to_skills_ratio',
 'text_density_index',
 'low_skills_high_urgency',
 'is_anonymous_recruiter',
 'readability_score',
 'fraud_phrase_score',
 'urgency_score',
 'contact_risk_score',
 'skill_richness_score',
 'readability_score',
 'word_count',
 'sentence_count',
 'average_word_length',
 'unique_word_count',
 'lexical_diversity']

In [24]:
# ============================================================
# SAVE FINAL TRAINING DATASET
# ============================================================

final_dataset.to_csv(

    "../data/final_model_dataset.csv",

    index=False

)

print(
    "final_model_dataset.csv saved successfully!"
)

final_model_dataset.csv saved successfully!
